In [37]:
!pip install -q -U google-genai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
os.environ["GOOGLE_API_KEY"] = ""

In [2]:
from google import genai

client = genai.Client()
MODEL = "gemini-2.5-flash"

# Sanity check
response = client.models.generate_content(model=MODEL, contents="Say: ready")
print("LLM status:", response.text.strip())

LLM status: Ready


In [3]:
def ask_llm(prompt: str):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text.strip()

In [4]:
def extract_json(text: str):
    """
    Extract the first JSON object or array from model output.
    Handles markdown code fences like ```json ... ```.
    """
    # Remove markdown fences if present
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if fenced:
        text = fenced.group(1).strip()

    # Try full parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first JSON object or array
    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))

    raise ValueError(f"No JSON found in LLM response:\n{text}")

## Section 1: Data Structures
Defines all dataclasses used to store story state. Sections 2 and 3 (story generation and compilation) will populate these.

| Class | Role | Function |
|---|---|---|
| `CrimeDetails` | The hidden truth | Keeps the ground-truth crime facts isolated so the LLM can never accidentally leak them into suspect dialogue or clue descriptions |
| `Character` | Any person in the story | Stores means/motive/opportunity explicitly so the meta-controller can check MMO coverage without re-querying the LLM |
| `Clue` | A piece of evidence | `is_red_herring` and `discovered` flags let the controller track what the detective *can* know vs. what *really* points to the criminal |
| `PlotPoint` | One story beat | `suspense_score` on each beat makes the rising tension arc a measurable, enforceable property rather than a vague stylistic goal |
| `StoryWorld` | Global state container | A single object passed by reference into every generation function — any LLM call can read current state via `summary_so_far()` without duplicating data |

In [5]:
import json
import re
from dataclasses import dataclass, field
from typing import List, Optional
from pprint import pprint


@dataclass
class Character:
    """A character in the story: detective, suspect, criminal, or witness."""
    name: str
    role: str                        # 'detective' | 'suspect' | 'criminal' | 'witness'
    description: str
    means: str                       # capability to commit the crime
    motive: str                      # reason to commit the crime
    opportunity: str                 # could they have been at the scene?
    alibi: str
    alibi_is_lie: bool = False
    is_criminal: bool = False
    eliminated: bool = False         # True once the detective rules them out
    notes: List[str] = field(default_factory=list)


@dataclass
class Clue:
    """A piece of evidence or information discoverable by the detective."""
    clue_id: int
    description: str
    location: str
    discovered: bool = False
    is_red_herring: bool = False
    points_to: str = ""              # character or event this implicates
    revealed_in_plot_point: int = -1


@dataclass
class PlotPoint:
    """One story beat that advances the plot."""
    index: int
    title: str
    description: str
    action_taken: str
    outcome: str
    obstacle: str
    suspense_score: float            # 0.0 (low tension) → 1.0 (maximum)
    characters_involved: List[str] = field(default_factory=list)
    clues_revealed: List[int] = field(default_factory=list)
    clues_used: List[int] = field(default_factory=list)


@dataclass
class CrimeDetails:
    """The hidden backstory: what really happened before the detective arrives."""
    crime_type: str
    victim: str
    location: str
    time_of_crime: str
    method: str
    criminal_name: str
    criminal_motive: str
    criminal_means: str
    key_evidence_hidden: List[str] = field(default_factory=list)
    backstory_summary: str = ""


@dataclass
class StoryWorld:
    """Master container for all story state."""
    crime_details: Optional[CrimeDetails] = None
    characters: List[Character] = field(default_factory=list)
    clues: List[Clue] = field(default_factory=list)
    plot_points: List[PlotPoint] = field(default_factory=list)
    countdown_deadline: str = ""
    countdown_consequence: str = ""
    final_narrative: str = ""

    def get_character(self, name: str) -> Optional[Character]:
        for c in self.characters:
            if c.name.lower() == name.lower():
                return c
        return None

    def get_clue(self, clue_id: int) -> Optional[Clue]:
        for cl in self.clues:
            if cl.clue_id == clue_id:
                return cl
        return None

    def discovered_clues(self) -> List[Clue]:
        return [cl for cl in self.clues if cl.discovered]

    def active_suspects(self) -> List[Character]:
        return [c for c in self.characters if c.role in ('suspect', 'criminal') and not c.eliminated]

    def summary_so_far(self) -> str:
        """Brief text digest of current story state, used as context in LLM prompts."""
        lines = []
        if self.crime_details:
            cd = self.crime_details
            lines.append(f"CRIME: {cd.crime_type} of {cd.victim} at {cd.location} ({cd.time_of_crime}).")
        lines.append(f"ACTIVE SUSPECTS: {[c.name for c in self.active_suspects()]}")
        lines.append(f"CLUES FOUND: {[cl.description[:40] for cl in self.discovered_clues()]}")
        if self.plot_points:
            last = self.plot_points[-1]
            lines.append(f"LAST PLOT POINT ({last.index}): {last.title} — {last.outcome}")
        return "\n".join(lines)


print("Data structures defined.")


Data structures defined.


In [6]:
# Checkpoint 1: confirm StoryWorld initializes correctly
world = StoryWorld()
print("StoryWorld initialized:", world)


StoryWorld initialized: StoryWorld(crime_details=None, characters=[], clues=[], plot_points=[], countdown_deadline='', countdown_consequence='', final_narrative='')


In [7]:
def build_crime_details_prompt() -> str:
    return """
You are helping build the hidden ground-truth backstory for a gala murder crime mystery.

Generate a single crime scenario for a detective story.

Requirements:
- The crime must be a serious crime suitable for a mystery story.
- It must be solvable, but not obvious.
- The crime must support multiple plausible suspects later.
- The criminal's motive, means, and opportunity must all make sense together.
- The method of the crime must leave behind evidence that can later be discovered.
- The crime should not rely on supernatural elements or impossible coincidences.

Return ONLY valid JSON with exactly these keys:
{
  "crime_type": "...",
  "victim": "...",
  "location": "...",
  "time_of_crime": "...",
  "method": "...",
  "criminal_name": "...",
  "criminal_motive": "...",
  "criminal_means": "...",
  "key_evidence_hidden": ["...", "...", "..."],
  "backstory_summary": "..."
}

Field guidance:
- crime_type: e.g. "murder", "blackmail", "kidnapping", "art theft" (in this case, it will be "murder")
- victim: full name and a few identifying words
- location: specific and story-friendly
- time_of_crime: specific enough to anchor a timeline
- method: how the crime was carried out
- criminal_name: full name
- criminal_motive: concrete and personal
- criminal_means: how they were capable of doing it
- key_evidence_hidden: 3 to 5 important hidden evidence items
- backstory_summary: 1 paragraph summary of what truly happened before the detective starts investigating

Important:
- The criminal should not be the most obvious suspect.
- The motive should be meaningful, not generic.
- The evidence should connect naturally to the crime method.
- Return JSON only. No explanation.
""".strip()

In [8]:
def generate_crime_details() -> CrimeDetails:
    prompt = build_crime_details_prompt()
    raw = ask_llm(prompt)
    data = extract_json(raw)

    crime = CrimeDetails(
        crime_type=data["crime_type"],
        victim=data["victim"],
        location=data["location"],
        time_of_crime=data["time_of_crime"],
        method=data["method"],
        criminal_name=data["criminal_name"],
        criminal_motive=data["criminal_motive"],
        criminal_means=data["criminal_means"],
        key_evidence_hidden=data.get("key_evidence_hidden", []),
        backstory_summary=data.get("backstory_summary", "")
    )
    return crime

In [9]:
world.crime_details = generate_crime_details()

pprint(world.crime_details)
print()
print(world.crime_details.backstory_summary)

CrimeDetails(crime_type='murder',
             victim='Elias Thorne, notorious real estate tycoon and host of '
                    "the annual 'Charity of the Stars' gala.",
             location="The Thorne Estate's secluded library, a room Elias "
                      'often retreated to for private conversations.',
             time_of_crime='Approximately 9:45 PM, just after the main '
                           'speeches and during the musical interlude.',
             method="The victim's customary specialized herbal tea, prepared "
                    'in an antique silver teapot, was laced with a fast-acting '
                    'neurotoxin designed to mimic a sudden cardiac arrest, '
                    'ingested shortly before he collapsed.',
             criminal_name='Clara Vance',
             criminal_motive="Elias Thorne had discovered Clara Vance's true "
                             'identity as the daughter of a man he had '
                             'ruthlessly

In [10]:
def build_characters_prompt(crime: CrimeDetails) -> str:
    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating the cast of characters for a crime mystery.

The hidden ground-truth crime is:
{json.dumps(crime_payload, indent=2)}

Create:
- 1 detective
- 5 suspects who are NOT the criminal
- 1 criminal character whose name MUST exactly match the criminal_name above

Return ONLY valid JSON with exactly this structure:
{{
  "detective": {{
    "name": "...",
    "role": "detective",
    "description": "...",
    "means": "N/A",
    "motive": "solve the case",
    "opportunity": "arrives after the crime to investigate",
    "alibi": "N/A"
  }},
  "criminal": {{
    "name": "...",
    "role": "criminal",
    "description": "...",
    "means": "...",
    "motive": "...",
    "opportunity": "...",
    "alibi": "...",
    "alibi_is_lie": true
  }},
  "suspects": [
    {{
      "name": "...",
      "role": "suspect",
      "description": "...",
      "means": "...",
      "motive": "...",
      "opportunity": "...",
      "alibi": "...",
      "alibi_is_lie": false
    }}
  ]
}}

Requirements:
- The criminal name must exactly match the given criminal_name.
- There must be exactly 5 suspects in the suspects list.
- Every suspect must have plausible means, motive, and opportunity.
- At least 2 suspects should look strongly suspicious at first.
- At least 2 innocent suspects should have a false or misleading alibi.
- The criminal should not be the most obvious suspect.
- The detective should feel capable and specific, not generic.
- Suspects should have relationships to the victim, location, or circumstances of the crime.
- Keep all details realistic and internally consistent.

Return JSON only.
""".strip()


def generate_characters(world: StoryWorld) -> List[Character]:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating characters.")

    raw = ask_llm(build_characters_prompt(world.crime_details))
    data = extract_json(raw)

    characters = []

    detective_data = data["detective"]
    detective = Character(
        name=detective_data["name"],
        role="detective",
        description=detective_data["description"],
        means=detective_data.get("means", "N/A"),
        motive=detective_data.get("motive", "solve the case"),
        opportunity=detective_data.get("opportunity", "arrives after the crime to investigate"),
        alibi=detective_data.get("alibi", "N/A"),
        alibi_is_lie=False,
        is_criminal=False,
        eliminated=False,
        notes=[]
    )
    characters.append(detective)

    criminal_data = data["criminal"]
    criminal = Character(
        name=criminal_data["name"],
        role="criminal",
        description=criminal_data["description"],
        means=criminal_data["means"],
        motive=criminal_data["motive"],
        opportunity=criminal_data["opportunity"],
        alibi=criminal_data["alibi"],
        alibi_is_lie=criminal_data.get("alibi_is_lie", True),
        is_criminal=True,
        eliminated=False,
        notes=[]
    )
    characters.append(criminal)

    for suspect_data in data["suspects"]:
        suspect = Character(
            name=suspect_data["name"],
            role="suspect",
            description=suspect_data["description"],
            means=suspect_data["means"],
            motive=suspect_data["motive"],
            opportunity=suspect_data["opportunity"],
            alibi=suspect_data["alibi"],
            alibi_is_lie=suspect_data.get("alibi_is_lie", False),
            is_criminal=False,
            eliminated=False,
            notes=[]
        )
        characters.append(suspect)

    return characters

    
def build_character_validation_prompt(world: StoryWorld, characters: List[Character]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
            }
            for c in characters
        ]
    }

    return f"""
You are validating the character set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Is there exactly 1 detective?
2. Is there exactly 1 criminal and does their name match the hidden criminal_name?
3. Are there exactly 5 non-criminal suspects?
4. Does every suspect have plausible means, motive, or opportunity?
5. Are at least 2 suspects genuinely suspicious at first?
6. Are the characters distinct from one another?
7. Are there any contradictions with the crime setup?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


def validate_characters(world: StoryWorld, characters: List[Character]) -> dict:
    raw = ask_llm(build_character_validation_prompt(world, characters))
    return extract_json(raw)

In [11]:
def build_clues_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist before generating clues.")
    if not world.characters:
        raise ValueError("world.characters must exist before generating clues.")

    crime = world.crime_details
    character_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "means": c.means,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "alibi_is_lie": c.alibi_is_lie,
            "is_criminal": c.is_criminal,
        }
        for c in world.characters
    ]

    crime_payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "criminal_means": crime.criminal_means,
        "key_evidence_hidden": crime.key_evidence_hidden,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are generating discoverable clues for a crime mystery.

Hidden crime details:
{json.dumps(crime_payload, indent=2)}

Character list:
{json.dumps(character_payload, indent=2)}

Generate 8 clues.

Return ONLY valid JSON with exactly this structure:
{{
  "clues": [
    {{
      "clue_id": 1,
      "description": "...",
      "location": "...",
      "is_red_herring": false,
      "points_to": "...",
      "why_it_matters": "..."
    }}
  ]
}}

Requirements:
- Generate exactly 8 clues.
- Clues must follow naturally from the true crime and the characters.
- At least 2 clues must initially point toward innocent suspects.
- At least 2 clues must meaningfully connect to the real criminal.
- At least 1 clue should relate to a false alibi.
- At least 1 clue should be ambiguous until later interpretation.
- The clues should not instantly solve the crime when seen alone.
- "points_to" should name a character or a specific event/fact.
- clue_id values must be 1 through 8.

Return JSON only.
""".strip()


def generate_clues(world: StoryWorld) -> List[Clue]:
    raw = ask_llm(build_clues_prompt(world))
    data = extract_json(raw)
    
    clues = []
    for clue_data in data["clues"]:
        clue = Clue(
            clue_id=clue_data["clue_id"],
            description=clue_data["description"],
            location=clue_data["location"],
            discovered=False,
            is_red_herring=clue_data.get("is_red_herring", False),
            points_to=clue_data.get("points_to", ""),
            revealed_in_plot_point=-1
        )
        clues.append(clue)
    
    return clues


def build_clue_validation_prompt(world: StoryWorld, clues: List[Clue]) -> str:
    crime = world.crime_details
    if not crime:
        raise ValueError("No crime details in world.")

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "criminal_name": crime.criminal_name,
            "criminal_motive": crime.criminal_motive,
            "criminal_means": crime.criminal_means,
            "backstory_summary": crime.backstory_summary,
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "is_criminal": c.is_criminal,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in clues
        ]
    }

    return f"""
You are validating the clue set for a crime mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Are there exactly 8 clues?
2. Do the clues arise naturally from the crime and character setup?
3. Do at least 2 clues point toward innocent suspects at first?
4. Do at least 2 clues connect meaningfully to the real criminal?
5. Is the mystery neither trivial nor impossible?
6. Are any clues redundant, contradictory, or disconnected?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


def validate_clues(world: StoryWorld, clues: List[Clue]) -> dict:
    raw = ask_llm(build_clue_validation_prompt(world, clues))
    return extract_json(raw)

In [12]:
def build_countdown_prompt(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details must exist first")

    crime = world.crime_details

    payload = {
        "crime_type": crime.crime_type,
        "victim": crime.victim,
        "location": crime.location,
        "time_of_crime": crime.time_of_crime,
        "method": crime.method,
        "criminal_name": crime.criminal_name,
        "criminal_motive": crime.criminal_motive,
        "backstory_summary": crime.backstory_summary,
    }

    return f"""
You are designing a countdown mechanism for a suspenseful detective story.

The countdown must:
- Be concrete and time-bound
- Be directly connected to the crime or investigation
- Create urgency that increases over time
- Make failure meaningful and irreversible

Crime context:
{json.dumps(payload, indent=2)}

Generate:
1. A specific countdown deadline (what event is approaching?)
2. A specific consequence if the detective fails before the deadline

Constraints:
- The deadline should feel natural to the scenario (not arbitrary)
- The consequence must be severe and story-relevant
- The consequence should make the truth harder or impossible to recover
- Avoid vague phrasing like "things get worse"

Examples of good countdowns:
- "By sunrise, the crime scene will be destroyed by the incoming storm."
- "In 24 hours, the prosecutor will formally charge the wrong suspect."
- "The suspect is scheduled to leave the country tonight."
- "A building demolition will erase crucial evidence."

Do not copy these exactly. Generate a new one specific to the crime.

Return ONLY valid JSON:
{{
  "countdown_deadline": "...",
  "countdown_consequence": "..."
}}
""".strip()


def generate_countdown(world: StoryWorld) -> StoryWorld:
    raw = ask_llm(build_countdown_prompt(world))
    data = extract_json(raw)

    world.countdown_deadline = data["countdown_deadline"]
    world.countdown_consequence = data["countdown_consequence"]

    return world

In [13]:
def populate_story_world(world: StoryWorld) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")

    characters = generate_characters(world)
    char_validation = validate_characters(world, characters)
    if not char_validation["is_valid"]:
        print("Invalid characters -- try again")
        pprint(char_validation)

    world.characters = characters

    clues = generate_clues(world)
    clue_validation = validate_clues(world, clues)
    if not clue_validation["is_valid"]:
        print("Invalid clues -- try again")
        pprint(clue_validation)

    world.clues = clues

    world = generate_countdown(world)
    return world

In [14]:
def print_characters(world: StoryWorld):
    print("\nCHARACTERS")
    print("=" * 60)
    for c in world.characters:
        print(f"Name: {c.name}")
        print(f"Role: {c.role}")
        print(f"Description: {c.description}")
        print(f"Means: {c.means}")
        print(f"Motive: {c.motive}")
        print(f"Opportunity: {c.opportunity}")
        print(f"Alibi: {c.alibi}")
        print(f"Alibi is lie: {c.alibi_is_lie}")
        print(f"Is criminal: {c.is_criminal}")
        print("-" * 60)


def print_clues(world: StoryWorld):
    print("\nCLUES")
    print("=" * 60)
    for cl in sorted(world.clues, key=lambda x: x.clue_id):
        print(f"Clue #{cl.clue_id}")
        print(f"Description: {cl.description}")
        print(f"Location: {cl.location}")
        print(f"Points to: {cl.points_to}")
        print(f"Red herring: {cl.is_red_herring}")
        print("-" * 60)


def print_countdown(world: StoryWorld):
    print("\nCOUNTDOWN SETUP")
    print("=" * 60)
    print(f"Deadline: {world.countdown_deadline}")
    print(f"Consequence: {world.countdown_consequence}")

In [15]:
world = populate_story_world(world)

print_characters(world)
print_clues(world)
print_countdown(world)


CHARACTERS
Name: Detective Inspector Vivian Holloway
Role: detective
Description: A sharp, no-nonsense investigator from the regional major crimes unit, known for her keen psychological insights and ability to piece together seemingly unrelated details. She has a reputation for being relentless and unswayed by the trappings of wealth and power, often seeing through the facades presented by the elite.
Means: N/A
Motive: solve the case
Opportunity: arrives after the crime to investigate
Alibi: N/A
Alibi is lie: False
Is criminal: False
------------------------------------------------------------
Name: Clara Vance
Role: criminal
Description: Elias Thorne's exceptionally efficient and outwardly demure personal assistant, known for her meticulous organization and calm demeanor. Beneath her composed exterior, she harbors a deep-seated secret and a burning resentment towards Thorne, a man who unknowingly held her fate in his cruel hands.
Means: As Thorne's personal assistant, she had unquest

In [16]:
def build_countdown_text(step_index: int, total_steps: int = 15) -> str:
    remaining = total_steps - step_index + 1

    if remaining > 10:
        urgency = "low but rising"
    elif remaining > 5:
        urgency = "moderate and increasingly serious"
    elif remaining > 2:
        urgency = "high"
    else:
        urgency = "extreme"

    return (
        f"COUNTDOWN: The detective has only {remaining} major investigative moves remaining "
        f"before the case reaches a point of no return. Urgency level: {urgency}."
    )

In [17]:
def build_plot_context(world: StoryWorld) -> str:
    if not world.crime_details:
        raise ValueError("world.crime_details is missing")

    crime = world.crime_details

    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found in world.characters")

    suspects = [c for c in world.characters if c.role in ("suspect", "criminal")]
    discovered_clues = world.discovered_clues()

    payload = {
        "crime_details": {
            "crime_type": crime.crime_type,
            "victim": crime.victim,
            "location": crime.location,
            "time_of_crime": crime.time_of_crime,
            "method": crime.method,
            "backstory_summary": crime.backstory_summary,
        },
        "detective": {
            "name": detective.name,
            "description": detective.description,
            "motive": detective.motive,
        },
        "active_suspects": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "eliminated": c.eliminated,
            }
            for c in suspects if not c.eliminated
        ],
        "discovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in discovered_clues
        ],
        "undiscovered_clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
            }
            for cl in world.clues if not cl.discovered
        ],
        "prior_plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ]
    }

    return json.dumps(payload, indent=2)

In [18]:
def build_next_plot_point_prompt(world: StoryWorld, step_index: int, total_steps: int = 15) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)
    if detective is None:
        raise ValueError("No detective found")

    countdown_text = build_countdown_text(step_index, total_steps)

    detective_goal = (
        f"{detective.name} must identify the true criminal behind the "
        f"{world.crime_details.crime_type} of {world.crime_details.victim} before the final countdown expires."
    )

    deadline = getattr(world, "countdown_deadline", "")
    consequence = getattr(world, "countdown_consequence", "")
    
    countdown_block = f"""
    DEADLINE:
    {deadline}
    
    CONSEQUENCE:
    {consequence}
    """

    return f"""
You are the suspense-planning controller for a crime mystery.

Your job is to generate exactly ONE next plot point in a 15-step detective story.

This story must use suspense generation:
1. The audience cares about the detective and the victim.
2. The detective has a concrete goal.
3. Failure has dire consequences.
4. Each new plot point makes success harder, more urgent, or less likely.
5. Suspense comes from narrowing options, increasing uncertainty, and reducing chances to avert failure.
6. Do NOT resolve the full case early.

Detective goal:
{detective_goal}

{countdown_block}

{countdown_text}

Current story state:
{build_plot_context(world)}

Rules for this next plot point:
- This is plot point #{step_index} of {total_steps}.
- The detective must take a concrete action.
- That action must produce a meaningful outcome, but not fully solve the case unless this is the final plot point.
- Introduce or intensify an obstacle.
- Suspense must generally rise over time.
- Reveal 0, 1, or 2 clues at this step.
- Use already discovered clues when relevant.
- A clue can be revealed only once.
- You may eliminate at most one suspect in a single step.
- The detective should not identify the real criminal before plot point 13.
- Plot points 13-15 should feel like endgame escalation.
- A false lead, contradiction, missing evidence, time pressure, or witness unreliability can be used as obstacles.
- Keep the action grounded in crime solving, not action-movie spectacle.

Return ONLY valid JSON with exactly this structure:
{{
  "index": {step_index},
  "title": "...",
  "description": "...",
  "action_taken": "...",
  "outcome": "...",
  "obstacle": "...",
  "suspense_score": 0.0,
  "characters_involved": ["..."],
  "clues_revealed": [1, 2],
  "clues_used": [1, 2],
  "suspect_eliminated": "Name or empty string",
  "countdown_note": "How the countdown pressure changed in this step"
}}

Constraints on suspense_score:
- It should usually increase over time.
- Early steps: around 0.25 to 0.45
- Middle steps: around 0.45 to 0.75
- Late steps: around 0.75 to 0.98

Return JSON only.
""".strip()

In [19]:
def apply_plot_point_to_world(world: StoryWorld, data: dict) -> PlotPoint:
    plot_point = PlotPoint(
        index=data["index"],
        title=data["title"],
        description=data["description"],
        action_taken=data["action_taken"],
        outcome=data["outcome"],
        obstacle=data["obstacle"],
        suspense_score=float(data["suspense_score"]),
        characters_involved=data.get("characters_involved", []),
        clues_revealed=data.get("clues_revealed", []),
        clues_used=data.get("clues_used", [])
    )

    # Mark newly revealed clues as discovered
    for clue_id in plot_point.clues_revealed:
        clue = world.get_clue(clue_id)
        if clue:
            clue.discovered = True
            clue.revealed_in_plot_point = plot_point.index

    # Eliminate suspect if applicable
    suspect_name = data.get("suspect_eliminated", "").strip()
    if suspect_name:
        character = world.get_character(suspect_name)
        if character and character.role in ("suspect", "criminal") and not character.is_criminal:
            character.eliminated = True
            character.notes.append(f"Eliminated in plot point {plot_point.index}")

    world.plot_points.append(plot_point)
    return plot_point

In [20]:
def build_plot_point_validation_prompt(world: StoryWorld, candidate: dict, total_steps: int = 15) -> str:
    return f"""
You are validating one plot point in a suspenseful detective story.

Current story state:
{build_plot_context(world)}

Candidate plot point:
{json.dumps(candidate, indent=2)}

Check:
1. Is the detective taking a concrete investigative action?
2. Does the plot point increase or maintain suspense appropriately?
3. Does it avoid solving the whole case too early?
4. Are clue reveals valid and non-contradictory?
5. Is any suspect elimination justified?
6. Is the obstacle meaningful?
7. Is this consistent with a countdown-based suspense structure?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."]
}}
""".strip()


def validate_plot_point(world: StoryWorld, candidate: dict) -> dict:
    raw = ask_llm(build_plot_point_validation_prompt(world, candidate))
    return extract_json(raw)

In [21]:
def generate_next_plot_point(world: StoryWorld, step_index: int, total_steps: int = 15, max_attempts: int = 3) -> PlotPoint:
    last_candidate = None

    for attempt in range(max_attempts):
        prompt = build_next_plot_point_prompt(world, step_index, total_steps)
        raw = ask_llm(prompt)
        candidate = extract_json(raw)
        last_candidate = candidate

        validation = validate_plot_point(world, candidate)

        if validation.get("is_valid", False):
            return apply_plot_point_to_world(world, candidate)

    # fallback: use the last candidate even if imperfect
    if last_candidate is None:
        raise ValueError(f"Failed to generate plot point {step_index}")

    return apply_plot_point_to_world(world, last_candidate)

In [22]:
def generate_plot_points(world: StoryWorld, total_steps: int = 15) -> StoryWorld:
    if not world.crime_details:
        raise ValueError("Generate crime details first.")
    if not world.characters:
        raise ValueError("Generate characters first.")
    if not world.clues:
        raise ValueError("Generate clues first.")

    world.plot_points = []

    for step_index in range(1, total_steps + 1):
        print(f"Generating plot point {step_index}/{total_steps}...")
        plot_point = generate_next_plot_point(world, step_index, total_steps)
        print(f"  -> {plot_point.title} (suspense={plot_point.suspense_score:.2f})")

    return world

In [23]:
def build_full_plot_validation_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in world.plot_points
        ],
        "remaining_active_suspects": [c.name for c in world.active_suspects()],
        "discovered_clues": [cl.clue_id for cl in world.discovered_clues()]
    }

    return f"""
You are validating the full suspense arc of a 15-point detective mystery.

Data:
{json.dumps(payload, indent=2)}

Check:
1. Do the 15 plot points form a coherent progression?
2. Does suspense generally rise?
3. Does the countdown mechanism feel meaningful?
4. Is the case not solved too early?
5. Are the final 3 plot points endgame escalation?
6. Are clue reveals paced well?
7. Are there any repetitive or filler beats?

Return ONLY valid JSON:
{{
  "is_valid": true,
  "issues": ["..."],
  "suggested_fixes": ["..."],
  "arc_score": 0.0
}}
""".strip()


def validate_full_plot_arc(world: StoryWorld) -> dict:
    raw = ask_llm(build_full_plot_validation_prompt(world))
    return extract_json(raw)

In [24]:
def print_plot_points(world: StoryWorld):
    print("\nPLOT POINTS")
    print("=" * 80)

    for p in sorted(world.plot_points, key=lambda x: x.index):
        print(f"[{p.index}] {p.title}")
        print(f"Description: {p.description}")
        print(f"Action Taken: {p.action_taken}")
        print(f"Outcome: {p.outcome}")
        print(f"Obstacle: {p.obstacle}")
        print(f"Suspense Score: {p.suspense_score:.2f}")
        print(f"Characters Involved: {p.characters_involved}")
        print(f"Clues Revealed: {p.clues_revealed}")
        print(f"Clues Used: {p.clues_used}")
        print("-" * 80)

In [25]:
world = generate_plot_points(world, total_steps=15)
print_plot_points(world)

arc_validation = validate_full_plot_arc(world)
print("\nFULL ARC VALIDATION")
pprint(arc_validation)

Generating plot point 1/15...
  -> First Trace: A Glitch in the System and a Subtle Stain (suspense=0.35)
Generating plot point 2/15...
  -> A Ghost of Poison and a Faltering Alibi (suspense=0.42)
Generating plot point 3/15...
  -> A Scorched Secret: The Blackmail's Echo (suspense=0.52)
Generating plot point 4/15...
  -> A Twisted Hand and a Doctor's Dark Secret (suspense=0.60)
Generating plot point 5/15...
  -> A Fork in the Trail: The Blackmail's New Whisper (suspense=0.68)
Generating plot point 6/15...
  -> Echoes of Ruin and a Shifting Alibi (suspense=0.72)
Generating plot point 7/15...
  -> A Digital Ghost and an Unseen Hand (suspense=0.76)
Generating plot point 8/15...
  -> The Ghost's Retreat and the Digital Wall (suspense=0.82)
Generating plot point 9/15...
  -> A Ghostly Venom: The Untraceable Signature (suspense=0.88)
Generating plot point 10/15...
  -> The Ghost in the Machine: A Financial Fingerprint (suspense=0.92)
Generating plot point 11/15...
  -> A Calculated Bluff and

## What Remains To Be Built

### Section 2 — Story Detail Construction (David)
Populate the `StoryWorld` dataclasses using an iterative LLM prompting loop.

**Phase 1 — Crime backstory** (`CrimeDetails`)
- Prompt the LLM to generate the hidden crime: victim, location, time, method, criminal, motive, means, and hidden evidence.
- Store result in `world.crime_details`.

**Phase 2 — Characters & clues** (`Character`, `Clue`)
- Prompt for: one detective, one criminal (posing as suspect), and 3 innocent suspects — each with explicit means, motive, opportunity, and alibi.
- At least one suspect should have a false alibi (for an unrelated secret).
- Prompt for 6 clues: 4 genuine (point to criminal), 2 red herrings (mislead to innocent suspect).
- Append all to `world.characters` and `world.clues`.

**Phase 3 — Iterative suspense loop** (`PlotPoint`) — *the core of the system*
- Run 16 iterations. Each iteration makes two sequential LLM calls:
  - **Stage A**: given story state so far, propose the detective's next concrete action.
  - **Stage B**: introduce an obstacle that makes success feel less likely (suspense rises).
- Pass `world.summary_so_far()` as context in every prompt so the LLM stays consistent.
- Assign a `suspense_score` that rises monotonically from ~0.15 → ~0.98 across all 16 beats.
- Mark clues as `discovered=True` when they are revealed in a plot point.
- Append each result as a `PlotPoint` to `world.plot_points`.


---

### Section 3 — Story Fine-Tuning (Chaeyoung)
Validate and assemble the raw materials into a final readable story.

- **Validation**: assert `len(world.plot_points) >= 15`, at least one criminal, one detective, one red herring, one false alibi, and monotonically increasing suspense scores.
- **Consistency check**: prompt the LLM to review all characters, clues, and plot point outcomes for contradictions.
- **Narrative assembly**: pass all 16 raw `PlotPoint.description` fields to the LLM and ask it to rewrite them as a single fluent prose story. Store in `world.final_narrative`.
- **Hidden story reveal**: separately print `world.crime_details.backstory_summary` — this is the second story the detective reconstructs at the end.


In [26]:
def validate_story_structure(world: StoryWorld) -> dict:
    issues = []

    # Plot points
    if len(world.plot_points) < 15:
        issues.append(f"Expected at least 15 plot points, found {len(world.plot_points)}.")

    # Character role checks
    detectives = [c for c in world.characters if c.role == "detective"]
    criminals = [c for c in world.characters if c.is_criminal or c.role == "criminal"]

    if len(detectives) < 1:
        issues.append("No detective found.")
    if len(criminals) < 1:
        issues.append("No criminal found.")

    # Red herring clue
    red_herrings = [cl for cl in world.clues if cl.is_red_herring]
    if len(red_herrings) < 1:
        issues.append("No red herring clue found.")

    # False alibi
    false_alibis = [c for c in world.characters if c.alibi_is_lie]
    if len(false_alibis) < 1:
        issues.append("No false alibi found.")

    # Monotonic suspense
    suspense_scores = [p.suspense_score for p in sorted(world.plot_points, key=lambda x: x.index)]
    for i in range(1, len(suspense_scores)):
        if suspense_scores[i] < suspense_scores[i - 1]:
            issues.append(
                f"Suspense score decreased from plot point {i} to {i+1}: "
                f"{suspense_scores[i-1]} -> {suspense_scores[i]}"
            )
            break

    return {
        "is_valid": len(issues) == 0,
        "issues": issues
    }

In [27]:
structure_report = validate_story_structure(world)
print("STRUCTURE VALIDATION")
pprint(structure_report)

STRUCTURE VALIDATION
{'is_valid': True, 'issues': []}


In [28]:
def build_consistency_check_prompt(world: StoryWorld) -> str:
    payload = {
        "crime_details": {
            "crime_type": world.crime_details.crime_type if world.crime_details else "",
            "victim": world.crime_details.victim if world.crime_details else "",
            "location": world.crime_details.location if world.crime_details else "",
            "time_of_crime": world.crime_details.time_of_crime if world.crime_details else "",
            "method": world.crime_details.method if world.crime_details else "",
            "criminal_name": world.crime_details.criminal_name if world.crime_details else "",
            "criminal_motive": world.crime_details.criminal_motive if world.crime_details else "",
            "criminal_means": world.crime_details.criminal_means if world.crime_details else "",
            "backstory_summary": world.crime_details.backstory_summary if world.crime_details else "",
        },
        "characters": [
            {
                "name": c.name,
                "role": c.role,
                "description": c.description,
                "means": c.means,
                "motive": c.motive,
                "opportunity": c.opportunity,
                "alibi": c.alibi,
                "alibi_is_lie": c.alibi_is_lie,
                "is_criminal": c.is_criminal,
                "eliminated": c.eliminated,
                "notes": c.notes,
            }
            for c in world.characters
        ],
        "clues": [
            {
                "clue_id": cl.clue_id,
                "description": cl.description,
                "location": cl.location,
                "discovered": cl.discovered,
                "is_red_herring": cl.is_red_herring,
                "points_to": cl.points_to,
                "revealed_in_plot_point": cl.revealed_in_plot_point,
            }
            for cl in world.clues
        ],
        "plot_points": [
            {
                "index": p.index,
                "title": p.title,
                "description": p.description,
                "action_taken": p.action_taken,
                "outcome": p.outcome,
                "obstacle": p.obstacle,
                "suspense_score": p.suspense_score,
                "characters_involved": p.characters_involved,
                "clues_revealed": p.clues_revealed,
                "clues_used": p.clues_used,
            }
            for p in sorted(world.plot_points, key=lambda x: x.index)
        ]
    }

    return f"""
You are checking a crime mystery story plan for contradictions and logic problems.

Review the following story materials:
{json.dumps(payload, indent=2)}

Check for:
1. Character contradictions
2. Clue contradictions
3. Impossible timeline issues
4. Plot point outcomes that conflict with later events
5. Eliminated suspects acting as if still viable later without explanation
6. Clues that appear without being revealed
7. Any mismatch between the hidden crime and the detective's reconstructed story
8. Whether the final solution points to the real criminal

Return ONLY valid JSON in exactly this format:
{{
  "is_consistent": true,
  "issues": [
    "...",
    "..."
  ],
  "summary": "Brief overall judgment."
}}
""".strip()

In [29]:
def run_consistency_check(world: StoryWorld) -> dict:
    raw = ask_llm(build_consistency_check_prompt(world))
    return extract_json(raw)

In [30]:
consistency_report = run_consistency_check(world)
print("CONSISTENCY CHECK")
pprint(consistency_report)

CONSISTENCY CHECK
{'is_consistent': True,
 'issues': [],
 'summary': 'The story plan is highly consistent and well-structured. The '
            "criminal's motive, means, and opportunity are clearly established "
            'and logically revealed through the clues and plot points. Red '
            'herrings are effectively introduced and eventually dismissed or '
            "bypassed as the true culprit's trail becomes clearer. The "
            'escalating obstacles, particularly the encryption and '
            'legal/bureaucratic delays, create a strong sense of urgency and '
            'suspense, leading to a satisfying and logical climax where the '
            'killer is brought to justice through a clever final bluff. No '
            'contradictions in characters, clues, timeline, or plot outcomes '
            'were found. The final solution precisely matches the '
            'predetermined criminal and crime details.'}


In [35]:
def build_narrative_assembly_prompt(world: StoryWorld) -> str:
    detective = next((c for c in world.characters if c.role == "detective"), None)

    characters_payload = [
        {
            "name": c.name,
            "role": c.role,
            "description": c.description,
            "motive": c.motive,
            "opportunity": c.opportunity,
            "alibi": c.alibi,
            "is_criminal": c.is_criminal
        }
        for c in world.characters
    ]

    clues_payload = [
        {
            "clue_id": cl.clue_id,
            "description": cl.description,
            "location": cl.location,
            "is_red_herring": cl.is_red_herring,
            "points_to": cl.points_to,
            "revealed_in_plot_point": cl.revealed_in_plot_point
        }
        for cl in world.clues
    ]

    plot_descriptions = [
        {
            "index": p.index,
            "title": p.title,
            "description": p.description,
            "action_taken": p.action_taken,
            "outcome": p.outcome,
            "obstacle": p.obstacle,
            "characters_involved": p.characters_involved,
            "clues_revealed": p.clues_revealed,
            "clues_used": p.clues_used
        }
        for p in sorted(world.plot_points, key=lambda x: x.index)
    ]

    payload = {
        "detective_name": detective.name if detective else "The detective",
        "crime_type": world.crime_details.crime_type if world.crime_details else "",
        "victim": world.crime_details.victim if world.crime_details else "",
        "location": world.crime_details.location if world.crime_details else "",
        "countdown_deadline": getattr(world, "countdown_deadline", ""),
        "countdown_consequence": getattr(world, "countdown_consequence", ""),
        "characters": characters_payload,
        "clues": clues_payload,
        "plot_points": plot_descriptions
    }

    return f"""
You are a skilled mystery novelist.

Write a complete mystery story based on the materials below.

Story materials:
{json.dumps(payload, indent=2)}

Instructions:
- Write a full, immersive narrative in the style of a mystery novel.
- Do NOT write a synopsis or summary.
- Expand the plot points into scenes with atmosphere, movement, reflection, and tension.
- Preserve the sequence of events and the underlying logic of the investigation.
- Use the characters consistently with their descriptions and roles.
- Weave clues naturally into the prose.
- Develop red herrings in a believable way.
- Maintain strong suspense throughout, with the countdown pressure growing more intense as the story progresses.
- The detective should gradually piece the truth together through observation, inference, and misdirection.
- Use polished, engaging prose and natural transitions.
- The ending should feel like a proper mystery revelation, with the detective identifying the true criminal in a satisfying way.
- Length is encouraged; a detailed and well-written story is better than a compressed one.

Literary style:
- Clear but literary prose
- Varied sentence structure
- Strong scene transitions
- Show, don’t just tell
- Dialogue should be used frequently and naturally throughout the story

Dialogue requirements:
- Include regular dialogue between characters (detective, suspects, witnesses)
- Conversations should drive the investigation forward
- Use dialogue to reveal clues, contradictions, and personality
- Avoid long stretches without dialogue
- Dialogue should feel natural, not overly formal or robotic
- Mix dialogue with action and internal thoughts

Do not:
- mention "plot point"
- mention "countdown mechanism"
- list clues explicitly
- write headings, bullet points, or outline labels

Output only the final story.
""".strip()

In [36]:
def assemble_final_narrative(world: StoryWorld) -> str:
    if len(world.plot_points) < 15:
        raise ValueError("Need at least 15 plot points before assembling final narrative.")

    raw = ask_llm(build_narrative_assembly_prompt(world))
    world.final_narrative = raw.strip()
    return world.final_narrative

In [ ]:
final_story = assemble_final_narrative(world)
print(world.final_narrative)